# Lesson 6: Mapping Emotions

## Overview

This is the culminating lesson of the unit. Over the last several lessons you have:

- **Lesson 3** — Loaded and cleaned raw Reddit data
- **Lesson 4** — Extracted place names using NER models and resolved them to coordinates with a geoparser
- **Lesson 5** — Scored each sentence for sentiment using VADER and RoBERTa

Now you have two things attached to each sentence: a **location** and an **emotion**. In this lesson you will aggregate those scores by place and put them on a map — and then critically evaluate what that map can and cannot tell you.

---


## 1. Load the Data

Load the sentiment dataset your team produced in Lesson 5.2. If the file is not in memory, load it from the pickle file your teammate committed to the repository.


In [7]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr
import numpy as np

try:
    df_reddit_sentiment_full
    print("✅ df_reddit_sentiment_full is already loaded in memory")
except NameError:
    print("📥 Loading from shared data folder...")
    try:
        df_reddit_sentiment_full = pd.read_pickle('data/jmu_reddit_sentiment_full.pickle')
        print(f"✅ Loaded {len(df_reddit_sentiment_full):,} rows")
    except FileNotFoundError:
        print("❌ File not found — check that a teammate completed Section 7 of Lesson 5.2")

df_reddit_sentiment_full.sample(5, random_state=42)

📥 Loading from shared data folder...
✅ Loaded 1,785 rows


,type,date,score,year_month,sentences,toponyms,place,latitude,longitude,feature_name,nltk_sentiment,roberta_neg,roberta_neu,roberta_pos,roberta_compound
1374,comment,2018-10-11 12:25:33,1,2018-10,The drive to east campus is not bad at all- ma...,None,Port-sur-Seille,48.90279,6.16131,None,0.6956,0.012616,0.085959,0.901425,0.812407
1080,comment,2020-08-05 12:44:14,2,2020-08,"Babylon, Taste of India (the one near East Cam...",[Babylon],East Campus Quadrangle,38.90611,-77.06972,None,0.8126,0.040382,0.179310,0.780308,0.607250
1519,post,2020-02-23 12:34:56,34,2020-02,But now I can clearly see the JMU sticker and ...,"[Jersey, Jersey]",Bailiwick of Jersey,49.21667,-2.11667,None,0.8042,0.385345,0.519700,0.094956,-0.139474
289,comment,2020-07-08 13:31:07,3,2020-07,But these men were literally soldiers in a for...,[U.S.],United States,39.76,-98.5,None,-0.3291,0.839557,0.154469,0.005973,-0.704821
990,post,2021-10-04 21:43:21,42,2021-10,TIL JMU's delegate tony wilt voted to ban prem...,[virginia],Virginia,37.54812,-77.44675,None,-0.5574,0.226450,0.746693,0.026857,-0.050558


## 2. The Pipeline — A Critical Review

Before we visualize anything, it is worth stepping back to review what each stage of the pipeline actually did — and where it could have gone wrong.

### Lesson 3: Data Cleaning
You loaded raw Reddit posts and split them into individual sentences. You cleaned up encoding issues, removed very short strings, and filtered out noise. This early stage determines what the data will look like later.

> **Limitation:** Sentence splitting is imperfect. A sentence that spans a line break or uses non-standard punctuation may have been cut in the wrong place, which would mislead the sentiment model.

### Lesson 4: Location Extraction
You ran two NER (Named Entity Recognition) models — spaCy and a transformer-based tagger — to identify place names in each sentence, then used a geoparser to convert those names to coordinates.

> **Limitation:** NER models confuse place names with other entities (people, organizations, common words). The geoparser resolves ambiguous names by population rank and other weights, which means "London" will almost always map to the UK even if another location is closer and the author the author meant something else. You corrected some of these manually — but not all of them.

### Lesson 5: Sentiment Analysis
You ran VADER (rule-based, fast) and RoBERTa (transformer, context-aware) on each sentence and compared their outputs. You found contradictions — sentences where the two models disagreed — and used them to understand each model's blind spots.

> **Limitation:** Both models were trained on general social media text. Reddit language, irony, sarcasm, and in-group references may confuse either model. Sentiment scores are averages across sentences — a place mentioned once in a very negative post will look "negative" even if 99% of posts about it are neutral.

---

Keep these limitations in mind as you interpret the map below.


## 4. Most Positive and Negative Sentiment by Location

Which places are mentioned in the most positive or negative sentences? The bar chart below compares the top 5 and bottom 5 cities by average RoBERTa sentiment score, filtered to locations with at least 3 posts so that single-sentence anomalies don't dominate.


In [8]:
city_sentiment_avg = (
    df_reddit_sentiment_full
    .groupby('place')['roberta_compound']
    .agg(['mean', 'count'])
    .reset_index()
)
city_sentiment_avg.columns = ['place', 'avg_sentiment', 'post_count']

city_sentiment_filtered = city_sentiment_avg[city_sentiment_avg['post_count'] >= 3].copy()

top_5_cities = city_sentiment_filtered.nlargest(5, 'avg_sentiment')
bottom_5_cities = city_sentiment_filtered.nsmallest(5, 'avg_sentiment')
top_bottom_cities = pd.concat([top_5_cities, bottom_5_cities])
top_bottom_cities['category'] = ['Top 5'] * 5 + ['Bottom 5'] * 5

fig = px.bar(
    top_bottom_cities.sort_values('avg_sentiment'),
    x='avg_sentiment', y='place', orientation='h',
    color='category',
    title='Top 5 vs Bottom 5 Cities by Average Sentiment Score',
    labels={'avg_sentiment': 'Average RoBERTa Sentiment Score', 'place': 'City', 'category': 'Ranking'},
    hover_data={'post_count': True, 'avg_sentiment': ':.3f'},
    color_discrete_map={'Top 5': '#2E8B57', 'Bottom 5': '#CD5C5C'},
    height=600
)
fig.add_vline(x=0, line_dash="dash", line_color="gray",
              annotation_text="Neutral", annotation_position="top")
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Average Sentiment: %{x:.3f}<br>Number of Posts: %{customdata[0]}<br><extra></extra>"
)
fig.show()

> 💡 **Critical Reflection:**
> - Do any of the "most negative" cities surprise you? Could the negativity be about a topic *associated* with a place rather than the place itself?
> - What does the `post_count >= 3` threshold do to the results? What would change if you set it to 10?


## 5. Aggregate Sentiment by Location

To plot on a map we need one row per place with a latitude, longitude, and average sentiment score. The cell below collapses the sentence-level data down to the place level.


In [9]:
df_reddit_place_sentiments = df_reddit_sentiment_full.groupby('place').agg(
    location_count=('place', 'size'),
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first'),
    sentences=('sentences', lambda x: list(x)),
    avg_roberta_compound=('roberta_compound', 'mean'),
).reset_index()

df_reddit_place_sentiments.sample(10, random_state=42)

,place,location_count,latitude,longitude,sentences,avg_roberta_compound
434,Stine,1,37.49524,-114.58889,[Not an alumni but I found it neat that the St...,0.687468
440,Subway,1,26.07205,-80.23203,[Meal plan changes invoke rage I thought the m...,-0.262419
6,Airport,2,22.31602,113.93663,"[""Airport"" was my shit., Airport, that's what ...",-0.336629
184,Hall Mountain,2,40.99729,-77.27803,[Maury is now Mountain Hall and Ashby is now V...,0.000550
78,Chicago,3,41.85003,-87.65005,[Twice as many black people died from black on...,0.048485
299,Mill,2,30.47318,-83.40015,"[Not saying you get tons of extra amenities, b...",0.230436
521,Ṭikar,5,27.4813,83.52467,"[P.S., P.S., P.S., P.S., P.S.]",0.021212
484,Vizslás,1,48.05171,19.8199,[Are we sure it's the guy with vizslas?],-0.005962
117,Duryea,1,41.34397,-75.73853,"[Babylon, Taste of India (the one near East Ca...",0.607250
137,Engeo,2,53.47231,9.13028,[The top floors of EnGeo and King hall are alw...,0.063711


## 6. Mapping Emotions — By Design

Each bubble on a sentiment map encodes several simultaneous decisions: what size means, what colour means, how far to zoom, which places to include or exclude. A map that takes 30 seconds to generate may take hours to design honestly.

### Two Maps. One Dataset. Opposite Stories.

Before learning how each individual decision works, here is what is at stake when you make them all at once.

The cell below produces two maps from the same JMU Reddit data. They support opposite conclusions. Neither changes a single number.

**Map A** is built with choices that amplify a negative reading:
- A dark basemap that primes the reader for bad news.
- No `color_continuous_midpoint` — the most invisible manipulation on this list. Plotly anchors the colour scale's neutral yellow at the **midpoint of the data range**, not at zero. If the data ranges from −0.3 to +0.6, the midpoint is +0.15. Any place with sentiment below +0.15 — including mildly *positive* places — renders in the red-to-yellow band. The map says "negative" when the data says "slightly less positive than average."
- A raw count size encoding with an inflated maximum, so one large location dominates visually.
- National zoom at `zoom=3`, where geoparser errors (posts resolved to cities in Europe or Asia) appear as real data points.
- No filter: every single-mention location is included.

**Map B** is built with choices that produce a calm, mostly-positive reading:
- A minimal light basemap.
- Categorical bucketing with a generous neutral band (±0.10), so most places near zero are labelled "Neutral" rather than pulled toward red or green.
- Quantile size classification: bubbles are balanced across four size classes rather than dominated by one outlier.
- Regional zoom (`zoom=7`) centred on the Shenandoah Valley — geoparser errors are out of frame.
- A filter of `min_count ≥ 5` removes noise.

> **Before running the cell, write down your prediction:** which map will look more dramatic? Which more credible? After running, identify: which design choice made the biggest difference, and which was least obvious before you knew to look for it?

In [14]:
import numpy as np

print("=" * 70)
print("  TWO MAPS. THE SAME DATA. OPPOSITE CONCLUSIONS.")
print("=" * 70)
print()

# ── Data prep ────────────────────────────────────────────────────────────
df_A = df_reddit_place_sentiments.copy()                          # Map A: everything
df_B = df_reddit_place_sentiments[
    df_reddit_place_sentiments['location_count'] >= 5             # Map B: high-confidence only
].copy()

# ── Map A: design choices that read as "students are negative" ───────────
#
# The key manipulation is the MISSING color_continuous_midpoint.
# Plotly anchors the neutral (yellow) colour at the midpoint of the
# data RANGE, not at zero.  If sentiment spans [-0.3, +0.6], the midpoint
# is +0.15 — so ANY place with sentiment below +0.15 (including mildly
# positive ones) renders red-to-yellow on the scale.
#
data_mid = (df_A['avg_roberta_compound'].min() + df_A['avg_roberta_compound'].max()) / 2
looks_negative = (df_A['avg_roberta_compound'] < data_mid).sum()
print(f"── Map A diagnostics ──")
print(f"  Sentiment range: {df_A['avg_roberta_compound'].min():.3f} → "
      f"{df_A['avg_roberta_compound'].max():.3f}")
print(f"  Without midpoint anchor, 'neutral' yellow sits at: {data_mid:.3f}")
print(f"  → {looks_negative}/{len(df_A)} locations appear orange/red even if they are mildly positive")
print()

fig_A = px.scatter_map(
    df_A,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"avg_roberta_compound": ":.3f", "location_count": True,
                "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn",
    # ← NO color_continuous_midpoint: neutral colour drifts to the data midpoint
    size_max=100,                               # ← one place will dominate visually
    map_style="carto-darkmatter",               # ← dark basemap, ominous tone
    center={"lat": 39.0, "lon": -89.0},         # ← national framing
    zoom=3,                                     # ← geoparser errors fully visible
    height=620,
    title="<b>Map A</b>: 'JMU Students Are Largely Negative About the Places They Discuss'<br>"
          "<sub>Dark map · no midpoint anchor · raw count sizes · no filter · national zoom</sub>"
)
fig_A.update_layout(margin={"r": 0, "t": 90, "l": 0, "b": 0})
fig_A.show()

# ── Map B: design choices that read as "sentiment is mostly positive" ────

NEUTRAL_THRESHOLD = 0.10   # generous neutral band — ±0.10 around zero

bins       = [-float('inf'), -NEUTRAL_THRESHOLD, NEUTRAL_THRESHOLD, float('inf')]
cat_labels = ['Negative', 'Neutral', 'Positive']
df_B['sentiment_category'] = pd.cut(
    df_B['avg_roberta_compound'], bins=bins, labels=cat_labels
)

# Quantile size classification: equal number of places per class
df_B['size_class'] = pd.qcut(
    df_B['location_count'], q=4, labels=[1, 2, 3, 4], duplicates='drop'
).astype(float)

neg = (df_B['sentiment_category'] == 'Negative').sum()
neu = (df_B['sentiment_category'] == 'Neutral').sum()
pos = (df_B['sentiment_category'] == 'Positive').sum()
print(f"── Map B diagnostics ──")
print(f"  Filtered to min_count ≥ 5: {len(df_B)} of {len(df_A)} locations kept")
print(f"  At threshold ±{NEUTRAL_THRESHOLD}: "
      f"Negative={neg}  Neutral={neu}  Positive={pos}")
print()

fig_B = px.scatter_map(
    df_B,
    lat="latitude", lon="longitude",
    size="size_class",
    color="sentiment_category",
    hover_name="place",
    hover_data={"avg_roberta_compound": ":.3f", "location_count": True,
                "size_class": False, "latitude": False, "longitude": False},
    color_discrete_map={'Negative': '#d62728', 'Neutral': '#aec7e8', 'Positive': '#2ca02c'},
    category_orders={"sentiment_category": cat_labels},
    size_max=40,
    map_style="carto-positron",                 # ← clean, neutral basemap
    center={"lat": 38.4, "lon": -79.0},         # ← Shenandoah Valley / campus context
    zoom=7,                                     # ← regional framing
    height=620,
    title="<b>Map B</b>: 'Most Places Around JMU Evoke Neutral or Positive Sentiment'<br>"
          "<sub>Light map · categorical buckets (±0.10 neutral band) · "
          "quantile sizes · min_count ≥ 5 · regional zoom</sub>"
)
fig_B.update_layout(margin={"r": 0, "t": 90, "l": 0, "b": 0})
fig_B.show()

print("=" * 70)
print("SAME DATA. SAME PIPELINE. SAME NUMBERS.")
print()
print("Design choices that differ between Map A and Map B:")
print()
print(f"  Midpoint anchor   A: none  (neutral colour sits at {data_mid:+.3f})")
print( "                    B: categorical (neutral = ±0.10)")
print()
print( "  Basemap           A: carto-darkmatter")
print( "                    B: carto-positron")
print()
print( "  Zoom / framing    A: national (zoom=3)  — geoparser errors visible")
print( "                    B: regional (zoom=7)  — campus context")
print()
print( "  Filter            A: min_count=1  (all locations, including noise)")
print( "                    B: min_count=5  (high-confidence locations only)")
print()
print( "  Size encoding     A: raw count, size_max=100 — one bubble dominates")
print( "                    B: quantile class, size_max=40 — balanced")
print()
print("💡 Neither map is 'wrong'. Both are defensible. But only one is appropriate")
print("   for your research question — and you must be able to say why.")
print("=" * 70)

  TWO MAPS. THE SAME DATA. OPPOSITE CONCLUSIONS.

── Map A diagnostics ──
  Sentiment range: -0.839 → 0.983
  Without midpoint anchor, 'neutral' yellow sits at: 0.072
  → 386/522 locations appear orange/red even if they are mildly positive



── Map B diagnostics ──
  Filtered to min_count ≥ 5: 62 of 522 locations kept
  At threshold ±0.1: Negative=20  Neutral=32  Positive=10



SAME DATA. SAME PIPELINE. SAME NUMBERS.

Design choices that differ between Map A and Map B:

  Midpoint anchor   A: none  (neutral colour sits at +0.072)
                    B: categorical (neutral = ±0.10)

  Basemap           A: carto-darkmatter
                    B: carto-positron

  Zoom / framing    A: national (zoom=3)  — geoparser errors visible
                    B: regional (zoom=7)  — campus context

  Filter            A: min_count=1  (all locations, including noise)
                    B: min_count=5  (high-confidence locations only)

  Size encoding     A: raw count, size_max=100 — one bubble dominates
                    B: quantile class, size_max=40 — balanced

💡 Neither map is 'wrong'. Both are defensible. But only one is appropriate
   for your research question — and you must be able to say why.


The six sections below show you exactly how each of those decisions works — one at a time. Work through all six, then fill in your design brief before writing your own map.

## 6.1 Map Design — Before You Touch the Code

You have the aggregated data. You could produce a working map in 30 seconds with the default settings. But every default encodes a claim:

- Default size (`size = post count`) makes the most-discussed place larger than everything else — is that intentional?
- Default colour scheme (`RdYlGn`, red–yellow–green) uses red and green — who cannot read that?
- Default zoom (`zoom=6`) frames Virginia — what disappears at the edges, and what is the effect of that absence?
- No filter (`location_count ≥ 1`) includes places mentioned once — are those meaningful?

The six sections below isolate each design variable. **Run each code cell, compare the versions, and answer the reflection questions before moving on.** Do not proceed to the design brief until you have worked through all six.

| # | Design Decision | The core question |
|---|---|---|
| 1 | **Bubble size** | What does size mean, and does it serve your argument? |
| 1b | **Size classification** | Equal interval, quantile, or Jenks natural breaks? |
| 2 | **Sentiment bucketing** | Continuous gradient or discrete categories? |
| 3 | **Color scale** | Which scale is honest *and* accessible? |
| 4 | **Base map & zoom** | What context do you show — and what do you hide? |
| 5 | **Filtering threshold** | Which places are worth showing? |

---

### Decision 1: Bubble Size

On a bubble map, the reader's eye is drawn to the largest circles first. **Size is an argument** — making something bigger implies it matters more. That means the moment you decide what drives bubble size, you have made a claim about what is important.

The default choice — `size = post count` — argues that places discussed more are more significant. That is often reasonable, but it creates a problem: a single heavily-discussed location can visually dominate the entire map, even if its sentiment is unremarkable.

| Encoding | What it emphasises | Risk |
|---|---|---|
| `size = raw count` (linear) | Frequency of discussion | One outlier can overwhelm everything else |
| `size = √count` (square root) | Frequency, compressed | Scale is less intuitive; readers may not know how to decode it |
| Uniform size | Nothing — only colour speaks | Makes it harder to see where the data is sparse vs. rich |

> 💡 **Questions to consider:**
> - With raw count, which location dominates? Is that the most important story in the data?
> - With uniform size, does the map become clearer or more confusing?
> - Is it ever *misleading* to use size as a redundant encoding (showing count *and* colour shows sentiment, but a large neutral bubble draws more attention than a small but strongly negative one)?

In [11]:
# Decision 1: Bubble size — what does SIZE encode?

import numpy as np

df_size = df_reddit_place_sentiments.copy()
df_size['sqrt_count'] = np.sqrt(df_size['location_count'])
df_size['uniform']    = 1

base = dict(
    lat="latitude", lon="longitude",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"location_count": True, "avg_roberta_compound": ":.3f",
                "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=460
)

for size_col, size_max, label in [
    ("location_count", 60, "A — size = raw count (linear): popular places visually dominate"),
    ("sqrt_count",     35, "B — size = √count: compressed scale, less outlier dominance"),
    ("uniform",        12, "C — uniform size: only colour carries information"),
]:
    fig = px.scatter_map(df_size, size=size_col, size_max=size_max, title=label, **base)
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("💡 In version A, which location dominates visually? "
      "Is that dominance the story you want to tell — or does it distract from a more interesting finding?")

💡 In version A, which location dominates visually? Is that dominance the story you want to tell — or does it distract from a more interesting finding?


### Decision 1b: Size Classification — Equal Interval, Quantile, or Jenks?

Once you've decided that bubble size should encode post count, a second question follows immediately: **how do you translate raw counts into visual sizes?**

Reddit post counts are almost always **right-skewed**: a handful of well-known places (the university town, the state capital) accumulate dozens or hundreds of mentions, while the majority of locations appear only once or twice. If you map the raw value directly, one giant bubble dominates and every other place collapses toward invisible.

One solution is to **classify** counts into a small number of discrete size classes. The three standard classification methods are:

| Method | How it works | Best when... |
|---|---|---|
| **Equal interval** | Divides the count *range* into bins of equal width (e.g., 0–25, 25–50, 50–75, 75–100) | Data is roughly uniformly distributed — rare for geographic counts |
| **Quantile** | Divides the *sorted data* so each bin holds the same number of places (e.g., 25th / 50th / 75th percentile breaks) | You want every size class to appear equally often on the map |
| **Jenks natural breaks** | Finds the threshold values that **minimise within-class variance** — breaks fall at genuine gaps in the data | Data has natural clusters (almost always true for post-count data) |

**Why equal interval usually fails here:** If most places have 1–5 mentions and one has 200, all but one place fall into the lowest bin. The largest bin contains a single point — which defeats the purpose of classification entirely.

**Jenks is usually the right choice** for this data. It finds where the *real* gaps are — for example, it might identify {1–3, 4–10, 11–40, 41+} as the natural clusters rather than dividing the range arithmetically.

> 💡 **Questions to consider:**
> - After running the cell, how many places land in the largest equal-interval bin?
> - Where do the Jenks break points fall? Do those values correspond to any intuitive groupings?
> - Does the visual differentiation between small, medium, and large bubbles improve with Jenks?

In [ ]:
# Decision 1b: Size classification — equal interval, quantile, Jenks natural breaks

import numpy as np

# First, examine the distribution of post counts
counts = df_reddit_place_sentiments['location_count']
print("── Post count distribution ──")
print(f"  min={counts.min()}, median={counts.median():.0f}, mean={counts.mean():.1f}, max={counts.max()}")
print(f"  Skewness: {counts.skew():.2f}  (>1 = strongly right-skewed)")
print(f"  Top 10 values: {sorted(counts.values, reverse=True)[:10]}")
print()

K = 4  # number of size classes — try changing to 3 or 5

# Method A: Equal interval — bins of equal WIDTH
df_ei = df_reddit_place_sentiments.copy()
print(f"── A: Equal Interval ({K} equal-width classes) ──")
print(pd.cut(counts, bins=K).value_counts().sort_index().to_string())
df_ei['size_class'] = pd.cut(counts, bins=K, labels=range(1, K + 1)).astype(float)

# Method B: Quantile — bins of equal COUNT (same number of places per class)
df_qt = df_reddit_place_sentiments.copy()
print(f"\n── B: Quantile ({K} classes, equal number of places per class) ──")
print(pd.qcut(counts, q=K, duplicates='drop').value_counts().sort_index().to_string())
df_qt['size_class'] = pd.qcut(counts, q=K, labels=range(1, K + 1), duplicates='drop').astype(float)

# Method C: Jenks natural breaks — bins at genuine gaps in the data
try:
    import mapclassify
    jnb = mapclassify.JenksNaturalBreaks(counts.values, k=K)
    df_jnb = df_reddit_place_sentiments.copy()
    df_jnb['size_class'] = (jnb.yb + 1).astype(float)
    print(f"\n── C: Jenks Natural Breaks ──")
    print(f"  Break points: {[round(b, 1) for b in jnb.bins]}")
    print(pd.Series(jnb.yb + 1).value_counts().sort_index()
            .rename(lambda i: f"Class {i}").to_string())
    c_label = "C — Jenks: breaks at genuine gaps in the data (minimises within-class variance)"
except ImportError:
    print("\n⚠️  mapclassify not installed. Run:  pip install mapclassify")
    print("   Showing Quantile as fallback for C.")
    df_jnb = df_qt.copy()
    c_label = "C — Jenks unavailable (mapclassify not installed); showing Quantile"

# Plot all three side by side
base = dict(
    lat="latitude", lon="longitude",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"location_count": True, "size_class": False,
                "avg_roberta_compound": ":.3f", "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    size_max=50, map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=460
)

for label, df_method in [
    ("A — Equal Interval: uniform bin width; dominated by skewed outliers", df_ei),
    ("B — Quantile: equal number of places per class; breaks may fall at arbitrary counts", df_qt),
    (c_label, df_jnb),
]:
    fig = px.scatter_map(df_method, size="size_class", title=label, **base)
    fig.update_layout(margin=dict(r=0, t=55, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("\n💡 In version A, how many places end up in the largest size class?")
print("   In Jenks, where do the break points fall — do those values make intuitive sense?")
print("   Which classification gives the most visual differentiation across the map?")

### Decision 2: Bucketing — Continuous or Categorical?

RoBERTa produces a score from roughly −1 (strongly negative) to +1 (strongly positive). You have two main options:

**Continuous** — map the raw score directly to a colour gradient. Every decimal of precision is preserved and a reader can compare −0.12 to −0.34 by shade. The tradeoff is that subtle differences in colour are hard to read at a glance.

**Categorical (bucketed)** — assign each place to Negative / Neutral / Positive based on a threshold you choose. The map becomes immediately legible to any reader, but:

- **The threshold is itself a claim.** A threshold of ±0.05 will call many more places "Neutral" than a threshold of ±0.2. There is no objectively correct cutoff — it depends on what you want to argue.
- **A place scoring −0.051 looks identical to one scoring −0.9.** Both are "Negative." The distinction between mild and severe disappears.
- **Variation within a category is invisible.** Two cities can appear the same red even if one is barely negative and the other is strongly negative.

> 💡 **Questions to consider:**
> - Which version makes your main finding easier to communicate to a general audience?
> - Adjust `NEUTRAL_THRESHOLD` in the code below. At what value does a place you care about flip category?
> - Is continuous or categorical more *honest* given what you know about the underlying data quality?

In [12]:
# Decision 2: Bucketing — continuous score vs. categorical labels

df_buck = df_reddit_place_sentiments.copy()

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    hover_name="place",
    hover_data={"avg_roberta_compound": ":.3f", "location_count": True,
                "latitude": False, "longitude": False},
    size_max=40, map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=490
)

# Version A: continuous — every decimal of precision preserved
print("── Version A: Continuous ──")
fig_A = px.scatter_map(
    df_buck, color="avg_roberta_compound",
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    title="A — Continuous: precise, but requires careful reading",
    **base
)
fig_A.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig_A.show()

# Version B: categorical — adjust NEUTRAL_THRESHOLD and re-run
NEUTRAL_THRESHOLD = 0.05   # ← try 0.01, 0.1, 0.2 and observe what changes

bins       = [-float('inf'), -NEUTRAL_THRESHOLD, NEUTRAL_THRESHOLD, float('inf')]
cat_labels = ['Negative', 'Neutral', 'Positive']
df_buck['bucket'] = pd.cut(df_buck['avg_roberta_compound'], bins=bins, labels=cat_labels)

counts = df_buck['bucket'].value_counts().sort_index()
print(f"\n── Version B: Three buckets  (neutral = ±{NEUTRAL_THRESHOLD}) ──")
print(counts.to_string())

fig_B = px.scatter_map(
    df_buck, color="bucket",
    color_discrete_map={'Negative': '#d62728', 'Neutral': '#aec7e8', 'Positive': '#2ca02c'},
    category_orders={"bucket": cat_labels},
    title=f"B — Categorical (threshold ±{NEUTRAL_THRESHOLD}): readable, but variation within each bucket is hidden",
    **base
)
fig_B.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig_B.show()

print(f"\n💡 Change NEUTRAL_THRESHOLD to 0.2 and re-run. How many places flip to 'Neutral'? "
      "What story does that version of the map tell?")

── Version A: Continuous ──



── Version B: Three buckets  (neutral = ±0.05) ──
bucket
Negative    156
Neutral     214
Positive    152



💡 Change NEUTRAL_THRESHOLD to 0.2 and re-run. How many places flip to 'Neutral'? What story does that version of the map tell?


### Decision 3: Color Scale

Color is the most powerful — and most easily manipulated — variable on a sentiment map.

**Diverging vs. sequential:**
Diverging scales (red↔green, red↔blue) are appropriate when zero is a meaningful midpoint and values exist on both sides. Setting `color_continuous_midpoint=0` anchors the neutral colour at zero. Sequential scales (light→dark) work better when all values fall on one side — they are not appropriate here unless you are mapping only one sentiment category.

**Cultural associations and accessibility:**
- **Red/green (RdYlGn)** is immediately intuitive because red=danger and green=safe are deeply ingrained. But red-green color blindness affects roughly 8% of men — for those readers, the entire argument of your map is invisible.
- **Red/blue (RdBu)** is more accessible and carries less cultural baggage about "good" and "bad."
- **Viridis** is fully colorblind-safe and perceptually uniform, but loses the positive/negative framing — every value looks like a point on a temperature scale rather than an emotional one.

**The midpoint is a claim:**
A diverging scale with its midpoint at 0 says: "zero means neutral." If your data skews positive (most scores above zero), shifting the midpoint upward could make the map look more balanced — or more negative — without changing a single data value.

> 💡 **Questions to consider:**
> - Who cannot read the RdYlGn map? What is your responsibility to that reader?
> - At what point does choosing a flattering color scale become misleading?
> - Should "neutral" be at the center of the color scale, or at the center of your data's range?

In [ ]:
# Decision 3: Color scale — four options compared

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    size_max=40, map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=440
)

scales = [
    ("RdYlGn  — default diverging (red=bad, green=good, accessibility issues)", "RdYlGn",  0),
    ("RdBu_r  — diverging, more accessible, less culturally loaded",            "RdBu_r",  0),
    ("Viridis — colorblind-safe, perceptually uniform, but loses pos/neg frame","Viridis", None),
    ("Spectral — high-contrast diverging",                                      "Spectral", 0),
]

for title, scale, midpoint in scales:
    kw = {"color_continuous_midpoint": midpoint} if midpoint is not None else {}
    fig = px.scatter_map(
        df_reddit_place_sentiments,
        color_continuous_scale=scale,
        title=title, **base, **kw
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
    fig.show()

print("💡 Which scale changes your emotional reaction to the map most? "
      "Does that reaction reflect the data — or the colors?")

### Decision 4: Base Map and Framing

The base map provides visual context — but it also makes implicit claims about geographic precision that the geoparsed data may not support.

**Base map style:**

| Style | Message it sends | Risk |
|---|---|---|
| Minimal / light (carto-positron) | Focus is on the data; geography is background | Strips context that might help readers orient themselves |
| Dark (carto-darkmatter) | High contrast; sentiment colours pop | Can make the map feel ominous regardless of the actual values |
| Full street map (OpenStreetMap) | Rich geographic context | Roads and labels compete with the data; implies more precision than geoparsing provides |

**Zoom and framing:**

Zoom level is a framing decision — it determines what is *in frame* and what is cut off. Zoom out to national scale and you will see geoparser errors: posts resolved to Paris, London, or cities in Asia. Zoom into Harrisonburg and those errors are out of frame — which makes the map cleaner, but also hides the fact that those errors exist.

> 💡 **Questions to consider:**
> - Does a dark base map change how you *feel* about the sentiment data even before reading a value?
> - Is choosing a tight zoom that hides geoparser errors honest? What should you disclose in a caption?
> - What geographic extent best serves your research question?

In [ ]:
# Decision 4: Base map style and zoom / framing

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    size_max=40, height=430
)

# Part A — base map style
print("── Part A: Three base map styles (same data, same zoom) ──")
for label, style in [
    ("Light — carto-positron  (minimal, data-focused)",    "carto-positron"),
    ("Dark — carto-darkmatter  (high contrast, dramatic)", "carto-darkmatter"),
    ("Standard — open-street-map  (maximum context)",      "open-street-map"),
]:
    fig = px.scatter_map(
        df_reddit_place_sentiments,
        map_style=style, center={"lat": 37.5, "lon": -78.0}, zoom=6,
        title=label, **base
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()

# Part B — zoom and framing
print("\n── Part B: Same data, three framing choices ──")
for zoom, lat, lon, label in [
    (3,  38.0,  -96.0,  "National (zoom=3) — geoparser errors are now visible"),
    (6,  37.5,  -78.0,  "Virginia (zoom=6) — regional framing"),
    (11, 38.44, -78.87, "Harrisonburg (zoom=11) — campus-level detail"),
]:
    fig = px.scatter_map(
        df_reddit_place_sentiments,
        map_style="carto-positron", center={"lat": lat, "lon": lon}, zoom=zoom,
        title=label, **base
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("\n💡 At national zoom, do any locations look obviously wrong? "
      "What would a reader conclude if they saw that map without knowing those were geoparser errors?")

### Decision 5: Filtering Threshold

Your dataset includes places mentioned just once. A single post about "Paris" — perhaps a student describing a holiday — appears on the map as a Paris data point. A higher minimum post count removes that noise, but it also makes less-discussed places disappear entirely. That is a form of **survivorship bias**: only popular places survive, and the map looks authoritative rather than partial.

| Threshold | Effect |
|---|---|
| `min_count = 1` | Every resolved location shown; maximum noise |
| `min_count = 3` | Removes single-post outliers; still sparse in some areas |
| `min_count = 10` | Clean signal; only frequently-mentioned places remain |

> 💡 **Questions to consider:**
> - At `min_count = 10`, which places survive? What do they have in common?
> - If a place the data *should* include disappears because it was only mentioned twice, is filtering still the right call?
> - What threshold does your **research question** justify?

In [ ]:
# Decision 5: Filtering threshold — how much data is enough?

for min_count in [1, 3, 10]:
    subset = df_reddit_place_sentiments[
        df_reddit_place_sentiments['location_count'] >= min_count
    ].copy()
    print(f"── min_count ≥ {min_count}: {len(subset)} locations, "
          f"{subset['location_count'].sum()} total posts ──")

    fig = px.scatter_map(
        subset,
        lat="latitude", lon="longitude",
        size="location_count",
        color="avg_roberta_compound",
        hover_name="place",
        color_continuous_scale="RdYlGn",
        color_continuous_midpoint=0,
        size_max=40,
        map_style="carto-positron",
        center={"lat": 37.5, "lon": -78.0},
        zoom=6, height=430,
        title=f"Filter: min post count ≥ {min_count}  →  {len(subset)} locations visible"
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("\n💡 Which location disappears first as you raise the threshold? "
      "Is its absence from your final map an honest choice?")

### 6.2 Design Brief

**Fill in this table before you write a single line of code.** A design brief forces every parameter to be a deliberate choice rather than an accepted default. Reference your observations from Decisions 1–5.

| Design Dimension | Your Choice | Reasoning |
|---|---|---|
| **Bubble size encoding** | Raw count / √count / uniform | |
| **Size classification** | Equal interval / Quantile / Jenks · Classes: ___ | |
| **Sentiment bucketing** | Continuous or categorical? If categorical, threshold: ___ | |
| **Color scale** | | |
| **Base map style** | | |
| **Center coordinates** | Lat: ___ · Lon: ___ | |
| **Zoom level** | | |
| **Minimum post count** | | |
| **Hover information** | What does a reader need to see? | |

> ⚠️ **Rule:** Your submitted map must differ from the reference implementation in at least three deliberate ways, each justifiable from this brief.

In [ ]:
# ============================================================
# YOUR FINAL MAP — build from your design brief above
# Every parameter must match a decision in your brief.
# ============================================================

fig = px.scatter_map(
    df_reddit_place_sentiments,
    lat="latitude",
    lon="longitude",
    # TODO: complete your design
)

fig.update_layout(
    # TODO: layout adjustments
)

fig.show()

### 6.3 Reference Implementation

The map below is one possible design using reasonable defaults. Study it critically before you submit your own version: which of the five decisions does it make, and are those the right choices for your research question?

After reviewing it, return to your own map cell above and revise where needed.

In [ ]:
# Reference implementation — study this critically, then go back and refine your own map above.
# You may NOT submit this cell's output unchanged as your final map.

fig = px.scatter_map(
    df_reddit_place_sentiments,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={'location_count': True, 'avg_roberta_compound': ':.3f', 'latitude': False, 'longitude': False},
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    size_max=50,
    title='Interactive Sentiment Map<br><sub>Bubble size = post count, Color = sentiment (red=negative, green=positive)</sub>',
    map_style="carto-positron",
    center={"lat": 37.5246322, "lon": -77.5758331},
    zoom=6, height=700
)
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>Posts: %{customdata[0]}<br>Avg Sentiment: %{customdata[1]}<br><extra></extra>"
)
fig.update_layout(
    margin={"r": 0, "t": 70, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        title="Average Sentiment",
        tickvals=[-0.5, -0.25, 0, 0.25, 0.5],
        ticktext=["Very Negative", "Negative", "Neutral", "Positive", "Very Positive"]
    )
)
fig.show()

print(f"📍 Total locations mapped: {len(df_reddit_place_sentiments)}")
print(f"📊 Total posts represented: {df_reddit_place_sentiments['location_count'].sum()}")
if len(df_reddit_place_sentiments) > 0:
    most_discussed = df_reddit_place_sentiments.loc[df_reddit_place_sentiments['location_count'].idxmax()]
    most_positive  = df_reddit_place_sentiments.loc[df_reddit_place_sentiments['avg_roberta_compound'].idxmax()]
    most_negative  = df_reddit_place_sentiments.loc[df_reddit_place_sentiments['avg_roberta_compound'].idxmin()]
    print(f"🔥 Most discussed: {most_discussed['place']} ({most_discussed['location_count']} posts)")
    print(f"😊 Most positive:  {most_positive['place']} (sentiment: {most_positive['avg_roberta_compound']:.3f})")
    print(f"😞 Most negative:  {most_negative['place']} (sentiment: {most_negative['avg_roberta_compound']:.3f})")

📍 Total locations mapped: 522
📊 Total posts represented: 1785
🔥 Most discussed: City of Harrisonburg (190 posts)
😊 Most positive:  Planetarium (sentiment: 0.983)
😞 Most negative:  Hampton Roads (sentiment: -0.839)


> 💡 **Critical Reflection:**
> - What patterns do you see geographically? Are certain regions consistently more positive or negative?
> - Find a location you know. Does the sentiment match your expectation? If not, what in the pipeline might explain it — data cleaning, NER, geoparsing, or the sentiment model?
> - What would the map look like if you used VADER scores instead of RoBERTa? What would be different?
> - This map represents what *Reddit users wrote about these places*, not what the places are actually like. What is the difference, and why does it matter?


## Lesson Summary

Here is what you covered in this lesson — and in the unit as a whole:

### The Full Pipeline
| Step | Lesson | Tool | What it produced |
|------|--------|------|-----------------|
| Load & clean data | 3 | Pandas | Sentence-level DataFrame |
| Extract place names | 4 | spaCy NER + transformer NER | Entity spans per sentence |
| Resolve to coordinates | 4 | Geoparser + GeoNames | Latitude/longitude per mention |
| Score sentiment | 5.1 | VADER | `vader_sentiment` per sentence |
| Score sentiment | 5.2 | RoBERTa | `roberta_compound` per sentence |
| Aggregate & map | 6 | Plotly | Emotional geography |

### Key Concepts
- **Named Entity Recognition (NER)** — identifying place names (and other entities) in unstructured text
- **Geoparsing** — disambiguating place names and resolving them to geographic coordinates
- **Sentiment analysis** — scoring text on a positive/negative scale; rule-based (VADER) vs. transformer-based (RoBERTa)
- **Aggregation** — collapsing row-level data to a summary by group (here: by place)
- **Limitations compound** — every imperfect step in a pipeline introduces noise that accumulates in the final output

### What the Map Does — and Doesn't — Show
This map visualises *how Reddit users wrote about places*, filtered through several imperfect models. It is not a ground-truth measure of how happy or unhappy those places are. That distinction — between the signal in the data and the reality it is meant to represent — is at the heart of critical data literacy.

---

➡️ **Next:** [Project: Mapping Emotions](../project_mapping_emotions/README.md)
